# Interesting Cases with Null Values in SDV

This cookbook explores specific scenarios you may encounter when synthesizing data with missing values. Each section is self-contained — jump to the case that's relevant to you.

> **Prerequisite:** This cookbook builds on concepts from *How to Generate Synthetic Data When Your Data Has Null Values*. If you're new to null handling in SDV, start there.

## Setup

We'll load our demo dataset and fit a default synthesizer that we can reference throughout the notebook.

## 1. When null rates don't match: `from_column` distortion

In the main cookbook, we saw that `'from_column'` mode can sometimes over- or under-produce nulls. Here we dig into *why* this happens and *how bad* it can get.

When you use `missing_value_generation='from_column'`, the synthesizer needs to model a binary indicator — "is this value null or not?" — alongside all other columns. The `GaussianCopulaSynthesizer` models everything using continuous Gaussian distributions, which creates a fundamental mismatch: it's trying to represent a 0/1 signal with a bell curve. The result is that the synthesizer can produce null rates that are wildly different from the original data.

**How bad can it get?**

*(Embedded bar chart will go here)*

The `response_time_hours` column is the most dramatic example: it jumps from 4.9% null in the real data to 100% null with `'from_column'`. Meanwhile, the default `'random'` mode stays within 1% of the real rate.

The severity of the distortion depends on the original null rate. Columns near 50% null tend to be modeled more accurately, while columns at the extremes (very few nulls or very many) are most susceptible to distortion.

> **Key takeaway:** If you use `'from_column'` with GaussianCopula, always compare `data.isnull().mean()` against `synthetic_data.isnull().mean()` after sampling. The default `'random'` mode is safer for null rate accuracy.

## 2. Columns that are entirely null

The `customer_notes` column in our dataset is 100% null — every single value is missing. What does SDV do with a column that contains no actual data?

**What happens when a column is 100% null?**

SDV handles this correctly — the synthetic output is also 100% null. Under the hood, the column gets encoded as random noise during preprocessing, but since the only known "category" is null, everything maps back to null during reverse transform.

This is expected behavior, but it means the synthesizer is doing unnecessary work modeling a column of pure noise. If you don't need the column in your synthetic output, consider dropping it from the data before fitting to simplify the synthesizer's task.

## 3. Preserving nullable integer types (`Int64`)

When a pandas integer column contains null values, pandas automatically converts it to `float64` — because the standard `int64` type can't represent `NaN`. This means values display as `1.0` instead of `1`, and nulls appear as `NaN` instead of `<NA>`.

Pandas offers a nullable integer type, `Int64` (with a capital I), that avoids this conversion. SDV can work with `Int64`, but it requires all three steps to round-trip correctly.

**Can SDV preserve `Int64`?**

To get nullable integers in synthetic output, you need:
1. **Cast the source column** to `Int64` via `.astype('Int64')`
2. **Set `computer_representation='Int64'`** in the metadata for that column
3. **Re-fit the synthesizer** after both changes

Without all three steps, integers degrade to `float64` and nulls show as `NaN` instead of `<NA>`. This matters if downstream systems expect integer types or if you're comparing data schemas between real and synthetic data.

## 4. How different synthesizers handle null values

SDV offers several single-table synthesizers: `GaussianCopulaSynthesizer`, `CTGANSynthesizer`, `TVAESynthesizer`, and `CopulaGANSynthesizer`. Since null handling happens in the preprocessing layer (before the data reaches the synthesizer), all of them share a common null handling pipeline.

**Does the synthesizer choice matter for nulls?**

The null rates are nearly identical across synthesizers. The small differences you see come from the synthesizers themselves (different modeling approaches produce slightly different distributions), not from how nulls are managed.

> **Key takeaway:** Your choice of synthesizer doesn't affect null handling. Pick the synthesizer based on your quality and performance needs — null behavior remains consistent.

## 5. How mean replacement affects synthetic data quality

Before the synthesizer trains, null values need to be replaced with actual numbers. The default strategy is `missing_value_replacement='mean'` — every null is filled with the column's mean value.

For columns with low null rates this barely matters. But for columns with high null rates, a significant portion of the training data becomes identical values clustered at the mean. The synthesizer then learns this artificial spike as a real pattern in the data.

**What does this look like?**

*(Embedded histogram will go here)*

With `response_time_hours` (~5% null), the effect is small — only 98 values are placed at the mean. But for a column like `satisfaction_score` (68% null), 68% of the training values would be clustered at a single point, significantly distorting the distribution the synthesizer learns.

The alternative is `missing_value_replacement='random'`, which fills nulls with a value chosen uniformly at random from the column's min/max range:

```python
FloatFormatter(missing_value_replacement='random', missing_value_generation='random')
```

This avoids the artificial spike and generally produces better synthetic data quality for high-null-rate columns.

## 6. What `None` mode does (and doesn't do) across column types

Setting `missing_value_generation=None` tells SDV not to recreate nulls for a given column. This works well for numerical and datetime columns — the synthetic output will have zero missing values. But not all column types respect this setting equally.

**Which columns still have nulls?**

The results show that several columns still have nulls despite `None` mode being configured on numerical and datetime columns:

- **Categorical columns** (`category`, `is_escalated`, `resolution_status`) — these treat null as just another category value, so null is part of their learned distribution regardless of the `missing_value_generation` setting
- **PII columns** (`customer_email`, `agent_name`, `customer_phone`) — these regenerate values using Faker and independently reintroduce nulls at the original rate
- **All-null columns** (`customer_notes`) — there's nothing to generate, so the column stays 100% null

To get truly zero nulls across *all* columns, you would need to either configure every column type individually or clean nulls from your source data before fitting.